In [61]:
import pandas as pd
import numpy as np
import os

In [62]:
your_flood_date = '2023-08-08 09:20:00'
flood_duration_days = 5
data_dir = '../../Data/Citizen Science/KV-Data'
metadata_file = "../../Data/Citizen Science/KV-Data/TB_Data_Summary.xlsx"
Target_projection = 'EPSG:32645'
#lat_long_bounds = [(27.6, 85.2), (27.8, 85.5)]  # Example bounds (min_lat, min_lon), (max_lat, max_lon)
output_location = '../../Results/Floods/Flood_Event'
if not os.path.exists(output_location):
    os.makedirs(output_location)


In [63]:
flood_peak_date = pd.to_datetime(your_flood_date)
flood_start_date = (flood_peak_date - pd.Timedelta(days=(flood_duration_days-1)/2)).strftime('%Y-%m-%d 00:00:00')
flood_end_date = (flood_peak_date + pd.Timedelta(days=(flood_duration_days-1)/2)).strftime('%Y-%m-%d 00:00:00')

In [64]:
from pyproj import Transformer

# Define transformer from UTM Zone 45N (EPSG:32645) to WGS84 (EPSG:4326)
transformer = Transformer.from_crs("EPSG:32645", "EPSG:4326", always_xy=True)

# Bounding box in UTM (meters)
x_min, y_min = 267396.45, 3021964.75
x_max, y_max = 408555.68, 3101900.27

# Transform UTM corners to lat/lon
lon_min, lat_min = transformer.transform(x_min, y_min)
lon_max, lat_max = transformer.transform(x_max, y_max)

# Create the lat_long_bounds in desired format: [[lat_min, lon_min], [lat_max, lon_max]]
lat_long_bounds = [[lat_min, lon_min], [lat_max, lon_max]]

print("lat_long_bounds =", lat_long_bounds)


lat_long_bounds = [[27.301011336452838, 84.64965824654722], [28.039258533841938, 86.0696410972238]]


In [65]:
#Read the excel file with citizen science metadata
metadata_citizen = pd.read_excel(metadata_file, sheet_name='TP_information')

#identify the station within the lat and long bounds
metadata_citizen = metadata_citizen[(metadata_citizen['Latitude'] >= lat_long_bounds[0][0]) &
                                    (metadata_citizen['Latitude'] <= lat_long_bounds[1][0]) &
                                    (metadata_citizen['Longitude'] >= lat_long_bounds[0][1]) &
                                    (metadata_citizen['Longitude'] <= lat_long_bounds[1][1])]

print(metadata_citizen.columns)

Index(['Logger_ID', 'Serial_Number', 'Site_Information', 'Latitude',
       'Longitude'],
      dtype='object')


In [66]:
# Read multiple Excel files and create precipitation data with Date/Time and stations as columns
import os
import glob
from pathlib import Path

def read_precipitation_data(data_directory, lat_long_bounds=None):
    """
    Read precipitation data from multiple Excel files and create a pivot table
    with Date and Time as first column and station names as column headers.
    
    Parameters:
    data_directory (str): Path to the directory containing Excel files
    lat_long_bounds (list): Optional bounds to filter stations [[lat_min, lon_min], [lat_max, lon_max]]
    
    Returns:
    DataFrame: Precipitation data with Date and Time as first column, stations as headers
    """
    
    # Get all Excel files except the summary file
    excel_files = glob.glob(os.path.join(data_directory, '*.xlsx'))
    excel_files = [f for f in excel_files if 'TB_Data_Summary' not in f]
    
    print(f"Found {len(excel_files)} data files to process")
    
    # Read the metadata to get station information
    metadata_file = os.path.join(data_directory, 'TB_Data_Summary.xlsx')
    metadata_all = pd.read_excel(metadata_file, sheet_name='TP_information')
    
    # Filter metadata by bounds if provided
    if lat_long_bounds:
        metadata_filtered = metadata_all[
            (metadata_all['Latitude'] >= lat_long_bounds[0][0]) &
            (metadata_all['Latitude'] <= lat_long_bounds[1][0]) &
            (metadata_all['Longitude'] >= lat_long_bounds[0][1]) &
            (metadata_all['Longitude'] <= lat_long_bounds[1][1])
        ]
        station_names_in_bounds = metadata_filtered['Logger_ID'].tolist()
        print(f"Stations within bounds: {station_names_in_bounds}")
    else:
        metadata_filtered = metadata_all
        station_names_in_bounds = metadata_all['Logger_ID'].tolist()
    
    # Initialize list to store precipitation dataframes
    all_precip_data = []
    
    # Process each Excel file
    for file_path in excel_files:
        filename = os.path.basename(file_path)
        station_id = filename.split(' ')[0] if ' ' in filename else filename.replace('.xlsx', '')
        
        # Check if this station is in our bounds
        if lat_long_bounds and station_id not in station_names_in_bounds:
            print(f"  Skipping {filename} - station {station_id} not in bounds")
            continue
            
        print(f"  Processing {filename} (Station: {station_id})")
        
        try:
            # Read all sheets from the Excel file
            xl_file = pd.ExcelFile(file_path)
            
            for sheet_name in xl_file.sheet_names:
                # Read the sheet
                df = pd.read_excel(file_path, sheet_name=sheet_name)
                
                # Keep only Date and Time and Precipitation columns
                if 'Date and Time' in df.columns and 'Precipitation (mm)' in df.columns:
                    precip_df = df[['Date and Time', 'Precipitation (mm)']].copy()
                    precip_df['Station_ID'] = station_id
                    
                    # Ensure Date and Time is datetime
                    precip_df['Date and Time'] = pd.to_datetime(precip_df['Date and Time'], errors='coerce')
                    
                    # Remove rows with missing datetime
                    precip_df = precip_df.dropna(subset=['Date and Time'])
                    
                    all_precip_data.append(precip_df)
                
        except Exception as e:
            print(f"  Error processing {filename}: {str(e)}")
    
    # Combine all precipitation data
    if all_precip_data:
        combined_precip = pd.concat(all_precip_data, ignore_index=True)
        
        # Create pivot table with Date and Time as index and stations as columns
        precipitation_pivot = combined_precip.pivot_table(
            index='Date and Time',
            columns='Station_ID', 
            values='Precipitation (mm)',
            aggfunc='first'  # In case there are duplicates, take the first value
        )
        
        # Reset index to make 'Date and Time' a regular column
        precipitation_pivot = precipitation_pivot.reset_index()
        
        # Sort by datetime
        precipitation_pivot = precipitation_pivot.sort_values('Date and Time')
        
        # Get station names with location info for better column names
        station_name_mapping = {}
        for station_id in [col for col in precipitation_pivot.columns if col != 'Date and Time']:
            station_info = metadata_filtered[metadata_filtered['Logger_ID'] == station_id]
            if not station_info.empty and 'Site_Information' in station_info.columns:
                location = station_info['Site_Information'].iloc[0]
                station_name_mapping[station_id] = f"{station_id}_{location}"
            else:
                station_name_mapping[station_id] = station_id
        
        # Rename columns with location info
        precipitation_pivot = precipitation_pivot.rename(columns=station_name_mapping)
        
        
        return precipitation_pivot, metadata_filtered
        
    else:
        print("No precipitation data was loaded.")
        return pd.DataFrame(), pd.DataFrame()

# Check if precipitation_data.csv already exists
csv_file_path = '../../Data/Citizen Science/precipitation_data.csv'

if os.path.exists(csv_file_path):
    print(f"📄 Loading existing precipitation data from: {csv_file_path}")
    precipitation_data = pd.read_csv(csv_file_path, parse_dates=['Date and Time'])
    print(f"✅ Loaded precipitation data: {precipitation_data.shape}")
    
    # Still need to create metadata_within_bounds for other cells
    print("\n📍 Creating metadata_within_bounds for coordinate transformations...")
    metadata_file_path = os.path.join(data_dir, 'TB_Data_Summary.xlsx')
    metadata_all = pd.read_excel(metadata_file_path, sheet_name='TP_information')
    
    # Filter metadata by bounds
    metadata_within_bounds = metadata_all[
        (metadata_all['Latitude'] >= lat_long_bounds[0][0]) &
        (metadata_all['Latitude'] <= lat_long_bounds[1][0]) &
        (metadata_all['Longitude'] >= lat_long_bounds[0][1]) &
        (metadata_all['Longitude'] <= lat_long_bounds[1][1])
    ]
    print(f"✅ Metadata loaded: {len(metadata_within_bounds)} stations within bounds")
    
else:
    print(f"📊 CSV file not found. Creating precipitation data from Excel files...")
    precipitation_data, metadata_within_bounds = read_precipitation_data(
        data_dir, 
        lat_long_bounds=lat_long_bounds
    )
    
    # Save the data
    precipitation_data.to_csv(csv_file_path, index=False)
    print(f"💾 Precipitation data saved to: {csv_file_path}")

📄 Loading existing precipitation data from: ../../Data/Citizen Science/precipitation_data.csv
✅ Loaded precipitation data: (247626, 9)

📍 Creating metadata_within_bounds for coordinate transformations...
✅ Metadata loaded: 9 stations within bounds


In [67]:
# Filter precipitation data for flood period and identify stations with complete data
def create_flood_period_data(precipitation_data, flood_start_date, flood_end_date, output_location):
    """
    Filter precipitation data for the specified flood period and identify stations 
    with complete data coverage or >95% coverage (with gap filling).
    
    Parameters:
    precipitation_data (DataFrame): The full precipitation dataset
    flood_start_date (str): Start date of flood period
    flood_end_date (str): End date of flood period
    output_location (str): Directory to save the output CSV
    
    Returns:
    DataFrame: Filtered data for flood period with complete stations only
    """
    
    # Convert flood dates to datetime
    start_date = pd.to_datetime(flood_start_date)
    end_date = pd.to_datetime(flood_end_date)
    
    print(f"🌊 FLOOD PERIOD ANALYSIS")
    print("="*60)
    print(f"Flood Start Date: {start_date}")
    print(f"Flood End Date: {end_date}")
    print(f"Flood Duration: {(end_date - start_date).days + 1} days")
    
    # Filter data for the flood period
    flood_mask = (precipitation_data['Date and Time'] >= start_date) & (precipitation_data['Date and Time'] <= end_date)
    flood_data = precipitation_data[flood_mask].copy()
    
    if flood_data.empty:
        print("❌ No data found for the specified flood period!")
        return pd.DataFrame()
    
    print(f"\n📊 Data available during flood period:")
    print(f"Total records in flood period: {len(flood_data):,}")
    print(f"Date range in data: {flood_data['Date and Time'].min()} to {flood_data['Date and Time'].max()}")
    
    # Get station columns (exclude Date and Time)
    station_cols = [col for col in flood_data.columns if col != 'Date and Time']
    
    # Function to calculate distance between stations using their coordinates
    def calculate_station_distances(station_metadata):
        """Calculate distances between all stations"""
        from geopy.distance import geodesic
        distances = {}
        
        for i, station1 in enumerate(station_metadata['Logger_ID']):
            distances[station1] = {}
            lat1 = station_metadata.iloc[i]['Latitude']
            lon1 = station_metadata.iloc[i]['Longitude']
            
            for j, station2 in enumerate(station_metadata['Logger_ID']):
                if station1 != station2:
                    lat2 = station_metadata.iloc[j]['Latitude']
                    lon2 = station_metadata.iloc[j]['Longitude']
                    dist = geodesic((lat1, lon1), (lat2, lon2)).kilometers
                    distances[station1][station2] = dist
        
        return distances
    
    # Get station distances for gap filling
    print(f"\n📍 Calculating station distances for gap filling...")
    try:
        station_distances = calculate_station_distances(metadata_within_bounds)
    except Exception as e:
        print(f"⚠️ Warning: Could not calculate distances - {e}")
        station_distances = {}
    
    # Analyze data completeness for each station during flood period
    station_completeness = {}
    complete_stations = []
    high_completeness_stations = []  # >95% but <100%
    
    print(f"\n🏥 Station Data Completeness Analysis:")
    print("-" * 60)
    
    for station in station_cols:
        total_flood_records = len(flood_data)
        valid_records = flood_data[station].notna().sum()
        completeness_percent = (valid_records / total_flood_records) * 100
        
        station_completeness[station] = {
            'total_records': total_flood_records,
            'valid_records': valid_records,
            'missing_records': total_flood_records - valid_records,
            'completeness_percent': completeness_percent
        }
        
        # Categorize stations by completeness
        if completeness_percent == 100.0:
            complete_stations.append(station)
            status = "✅ COMPLETE"
        elif completeness_percent >= 95.0:
            high_completeness_stations.append(station)
            status = f"🔶 HIGH ({completeness_percent:.1f}%) - FILLABLE"
        else:
            status = f"❌ INCOMPLETE ({completeness_percent:.1f}%)"
        
        station_short = station.split('_')[0]
        print(f"{station_short:<8} | {valid_records:>6}/{total_flood_records:<6} records | {completeness_percent:>6.1f}% | {status}")
    
    # Function to fill missing data using nearby stations
    def fill_missing_data(flood_data, station_col, station_distances):
        """Fill missing data for a station using inverse distance weighting from nearby stations"""
        station_id = station_col.split('_')[0]
        
        if station_id not in station_distances:
            print(f"  ⚠️ No distance data for {station_id}, skipping gap filling")
            return flood_data[station_col]
        
        filled_data = flood_data[station_col].copy()
        missing_indices = filled_data.isna()
        
        if not missing_indices.any():
            return filled_data
        
        print(f"  🔧 Filling {missing_indices.sum()} missing values for {station_id}")
        
        # Get available nearby stations sorted by distance
        nearby_stations = []
        for other_station_id, distance in station_distances[station_id].items():
            # Find the full column name for this station
            other_station_col = None
            for col in station_cols:
                if col.startswith(other_station_id + '_'):
                    other_station_col = col
                    break
            
            if other_station_col and other_station_col in flood_data.columns:
                # Check if this station has good data coverage
                other_completeness = flood_data[other_station_col].notna().sum() / len(flood_data) * 100
                if other_completeness >= 80:  # Only use stations with >80% completeness
                    nearby_stations.append((other_station_col, distance))
        
        # Sort by distance (closest first)
        nearby_stations.sort(key=lambda x: x[1])
        
        if not nearby_stations:
            print(f"    ⚠️ No suitable nearby stations found for {station_id}")
            return filled_data
        
        print(f"    📍 Using {len(nearby_stations[:3])} nearest stations for gap filling")
        
        # For each missing value, use inverse distance weighting
        for idx in flood_data.index[missing_indices]:
            weights = []
            values = []
            
            # Use up to 3 nearest stations
            for station_col_nearby, distance in nearby_stations[:3]:
                value = flood_data.loc[idx, station_col_nearby]
                if pd.notna(value):
                    # Inverse distance weighting (add small value to avoid division by zero)
                    weight = 1.0 / (distance + 0.1)
                    weights.append(weight)
                    values.append(value)
            
            if weights:
                # Calculate weighted average
                weighted_value = sum(w * v for w, v in zip(weights, values)) / sum(weights)
                filled_data.loc[idx] = weighted_value
        
        filled_count = filled_data.notna().sum() - flood_data[station_col].notna().sum()
        print(f"    ✅ Filled {filled_count} values for {station_id}")
        
        return filled_data
    
    # Fill missing data for high completeness stations (>95%)
    print(f"\n🔧 FILLING MISSING DATA FOR HIGH COMPLETENESS STATIONS:")
    print("-" * 60)
    
    filled_stations = complete_stations.copy()  # Start with already complete stations
    
    for station in high_completeness_stations:
        station_id = station.split('_')[0]
        print(f"Processing {station_id}...")
        
        try:
            filled_column = fill_missing_data(flood_data, station, station_distances)
            
            # Check if filling was successful (should now be 100% complete)
            new_completeness = filled_column.notna().sum() / len(filled_column) * 100
            
            if new_completeness >= 99.0:  # Allow for slight rounding
                flood_data[station] = filled_column
                filled_stations.append(station)
                print(f"  ✅ {station_id} successfully filled to {new_completeness:.1f}% completeness")
            else:
                print(f"  ❌ {station_id} filling unsuccessful ({new_completeness:.1f}%)")
                
        except Exception as e:
            print(f"  ❌ Error filling {station_id}: {e}")
    
    print(f"\n📋 SUMMARY:")
    print(f"Stations with COMPLETE data (100%): {len(complete_stations)}")
    print(f"Stations with HIGH completeness (>95%): {len(high_completeness_stations)}")
    print(f"Stations successfully filled: {len(filled_stations) - len(complete_stations)}")
    print(f"Total usable stations: {len(filled_stations)}")
    print(f"Stations with INCOMPLETE data (<95%): {len(station_cols) - len(complete_stations) - len(high_completeness_stations)}")
    
    if not filled_stations:
        print("❌ No stations have complete data for the entire flood period!")
        return pd.DataFrame()
    
    print(f"\n✅ Stations with complete/filled flood period data:")
    for station in filled_stations:
        station_id = station.split('_')[0]
        location = station.split('_', 1)[1] if '_' in station else 'Unknown location'
        original_completeness = station_completeness[station]['completeness_percent']
        status = "Original" if station in complete_stations else f"Filled from {original_completeness:.1f}%"
        print(f"   • {station_id}: {location} ({status})")
    
    # Create filtered dataset with only complete/filled stations
    complete_columns = ['Date and Time'] + filled_stations
    flood_complete_data = flood_data[complete_columns].copy()
    
    # Sort by date
    flood_complete_data = flood_complete_data.sort_values('Date and Time')
    
    # Create filename based on flood dates
    start_year_month = start_date.strftime('%Y%m')
    end_year_month = end_date.strftime('%Y%m')
    filename = f"prec_{start_year_month}_{end_year_month}.csv"
    filepath = os.path.join(output_location, filename)
    
    # Save to CSV
    flood_complete_data.to_csv(filepath, index=False)
    
    print(f"\n💾 FLOOD DATA SAVED:")
    print(f"File: {filepath}")
    print(f"Shape: {flood_complete_data.shape}")
    print(f"Columns: {list(flood_complete_data.columns)}")
    
    # Display sample of the data
    print(f"\n📄 Sample of flood period data (first 10 rows):")
    print(flood_complete_data.head(10).to_string(index=False))
    
    return flood_complete_data

# Execute the flood period analysis
print("Starting flood period data analysis...")
flood_data_complete = create_flood_period_data(
    precipitation_data, 
    flood_start_date,
    flood_end_date, 
    output_location
)

if not flood_data_complete.empty:
    print(f"\n🎉 SUCCESS: Flood period data created with {flood_data_complete.shape[1]-1} complete stations!")
else:
    print(f"\n⚠️  WARNING: No complete data available for the flood period.")

Starting flood period data analysis...
🌊 FLOOD PERIOD ANALYSIS
Flood Start Date: 2023-08-06 00:00:00
Flood End Date: 2023-08-10 00:00:00
Flood Duration: 5 days

📊 Data available during flood period:
Total records in flood period: 385
Date range in data: 2023-08-06 00:00:00 to 2023-08-10 00:00:00

📍 Calculating station distances for gap filling...

🏥 Station Data Completeness Analysis:
------------------------------------------------------------
TP001    |    385/385    records |  100.0% | ✅ COMPLETE
TP002    |    385/385    records |  100.0% | ✅ COMPLETE
TP003    |    385/385    records |  100.0% | ✅ COMPLETE
TP004    |      0/385    records |    0.0% | ❌ INCOMPLETE (0.0%)
TP005    |    385/385    records |  100.0% | ✅ COMPLETE
TP006    |    385/385    records |  100.0% | ✅ COMPLETE
TP007    |    385/385    records |  100.0% | ✅ COMPLETE
TP008    |    385/385    records |  100.0% | ✅ COMPLETE

🔧 FILLING MISSING DATA FOR HIGH COMPLETENESS STATIONS:
--------------------------------------

In [68]:
# Calculate total precipitation for each station during the flood event
if not flood_data_complete.empty:
    print("📊 TOTAL PRECIPITATION PER STATION DURING FLOOD EVENT")
    print("=" * 60)
    
    # Clean date format (remove comma if present)
    clean_start = flood_start_date.lstrip(',')
    clean_end = flood_end_date.lstrip(',')
    print(f"Period: {clean_start} to {clean_end}")
    print(f"Duration: {(pd.to_datetime(clean_end) - pd.to_datetime(clean_start)).days + 1} days")
    
    # Get station columns (exclude Date and Time)
    station_cols = [col for col in flood_data_complete.columns if col != 'Date and Time']
    
    # Calculate total precipitation for each station
    station_totals = []
    
    for station in station_cols:
        total_precip = flood_data_complete[station].sum()
        max_precip = flood_data_complete[station].max()
        non_zero_count = (flood_data_complete[station] > 0).sum()
        station_id = station.split('_')[0]
        location = station.split('_', 1)[1] if '_' in station else 'Unknown location'
        
        station_totals.append({
            'Station_ID': station_id,
            'Location': location,
            'Total_Precipitation_mm': round(total_precip, 2),
            'Max_15min_mm': round(max_precip, 2),
            'Rainy_Periods': non_zero_count
        })
    
    # Create DataFrame and sort by total precipitation (highest first)
    totals_df = pd.DataFrame(station_totals)
    totals_df = totals_df.sort_values('Total_Precipitation_mm', ascending=False)
    
    # Display the results
    print(f"\n{totals_df.to_string(index=False)}")
    
    # Summary statistics
    total_rain = totals_df['Total_Precipitation_mm'].sum()
    avg_rain = totals_df['Total_Precipitation_mm'].mean()
    max_station = totals_df.iloc[0] if not totals_df.empty else None
    
    print(f"\n📈 SUMMARY STATISTICS:")
    if total_rain > 0:
        print(f"Highest precipitation: {totals_df['Total_Precipitation_mm'].max():.2f} mm ({totals_df.iloc[0]['Station_ID']})")
        print(f"Lowest precipitation: {totals_df['Total_Precipitation_mm'].min():.2f} mm ({totals_df.iloc[-1]['Station_ID']})")
        print(f"Average precipitation: {avg_rain:.2f} mm")
        print(f"Total across all stations: {total_rain:.2f} mm")
    else:
        print("⚠️  NO PRECIPITATION recorded during this period")
        print("This appears to be a DRY PERIOD - consider selecting a different date range")
    
    print(f"Total stations analyzed: {len(totals_df)}")
    
    print(f"\n✅ Total precipitation calculated for all {len(station_totals)} stations!")
else:
    print("❌ No flood data available for precipitation totals calculation!")

📊 TOTAL PRECIPITATION PER STATION DURING FLOOD EVENT
Period: 2023-08-06 00:00:00 to 2023-08-10 00:00:00
Duration: 5 days

Station_ID                     Location  Total_Precipitation_mm  Max_15min_mm  Rainy_Periods
     TP007                      Bhardev                   142.6          13.0             92
     TP008       Okhreni Tipping Bucket                   127.4          15.0            114
     TP002 Tokha (Ashish Dangol's house                   124.8           9.2             75
     TP001   Nagarjun (Prativa's house)                   120.0           7.8             81
     TP006              Bhaktapur (KEC)                    89.4           6.8             73
     TP005               Kusunti Office                    57.4           3.4             74
     TP003                   Lapsephedi                     0.4           0.2              2

📈 SUMMARY STATISTICS:
Highest precipitation: 142.60 mm (TP007)
Lowest precipitation: 0.40 mm (TP003)
Average precipitation: 94.57 mm


In [69]:
# Convert flood period data to GSSHA rainfall input format
def create_gssha_rainfall_input(flood_data, metadata_within_bounds, target_projection, output_location, 
                               flood_start_date, flood_end_date):
    """
    Convert precipitation data to GSSHA rainfall input format.
    
    Parameters:
    flood_data (DataFrame): Flood period precipitation data with complete stations
    metadata_within_bounds (DataFrame): Station metadata with coordinates
    target_projection (str): Target projection (e.g., 'EPSG:32645')
    output_location (str): Directory to save the GSSHA input file
    flood_start_date (str): Start date of flood period
    flood_end_date (str): End date of flood period
    
    Returns:
    str: Path to the created GSSHA input file
    """
    
    if flood_data.empty:
        print("❌ No flood data available for GSSHA conversion!")
        return None
    
    print(f"🔧 CONVERTING TO GSSHA RAINFALL INPUT FORMAT")
    print("="*70)
    
    # Get station columns (exclude Date and Time)
    station_cols = [col for col in flood_data.columns if col != 'Date and Time']
    
    # Convert coordinates to target projection (UTM)
    from pyproj import Transformer
    
    # Create transformer from WGS84 to target projection
    coord_transformer = Transformer.from_crs("EPSG:4326", target_projection, always_xy=True)
    
    # Get station coordinates in UTM
    station_coords = {}
    station_descriptions = {}
    
    print(f"📍 Station Coordinates (converted to {target_projection}):")
    print("-" * 60)
    
    for station_col in station_cols:
        # Extract station ID from column name
        station_id = station_col.split('_')[0]
        
        # Find station in metadata
        station_info = metadata_within_bounds[metadata_within_bounds['Logger_ID'] == station_id]
        
        if not station_info.empty:
            lat = station_info['Latitude'].iloc[0]
            lon = station_info['Longitude'].iloc[0]
            site_info = station_info['Site_Information'].iloc[0] if 'Site_Information' in station_info.columns else station_id
            
            # Convert to UTM coordinates
            easting, northing = coord_transformer.transform(lon, lat)
            
            station_coords[station_col] = (easting, northing)
            station_descriptions[station_col] = f"{station_id} - {site_info}"
            
            print(f"  {station_id:<8} | {easting:>12.2f} {northing:>12.2f} | {site_info}")
        else:
            print(f"  ❌ Warning: No metadata found for {station_id}")
    
    # Calculate number of gages and time periods
    nrgag = len(station_coords)
    nrpds = len(flood_data)
    
    print(f"\n📊 GSSHA Parameters:")
    print(f"  Number of rain gages (NRGAG): {nrgag}")
    print(f"  Number of time periods (NRPDS): {nrpds}")
    print(f"  Time range: {flood_data['Date and Time'].min()} to {flood_data['Date and Time'].max()}")
    
    # Create GSSHA input file content
    gssha_content = []
    
    # EVENT header
    start_date_str = pd.to_datetime(flood_start_date).strftime('%Y-%m-%d')
    end_date_str = pd.to_datetime(flood_end_date).strftime('%Y-%m-%d')
    event_description = f"Flood Event {start_date_str} to {end_date_str} - Citizen Science Data"
    
    gssha_content.append(f'EVENT "{event_description}"')
    gssha_content.append(f"NRPDS {nrpds}")
    gssha_content.append(f"NRGAG {nrgag}")
    
    # COORD cards for each station
    for station_col, (easting, northing) in station_coords.items():
        description = station_descriptions[station_col]
        gssha_content.append(f'COORD {easting:.2f} {northing:.2f} "{description}"')
    
    # GAGES cards for each time period
    print(f"\n📝 Creating GAGES cards for {nrpds} time periods...")
    
    for idx, row in flood_data.iterrows():
        # Format date and time for GSSHA (YYYY MM DD HH MM)
        dt = row['Date and Time']
        date_str = f"{dt.year:04d} {dt.month:02d} {dt.day:02d} {dt.hour:02d} {dt.minute:02d}"
        
        # Get precipitation values for all stations at this time
        precip_values = []
        for station_col in station_coords.keys():
            # Convert from mm (15-min accumulation) to mm (keeping as accumulation for GAGES format)
            precip_val = row[station_col]
            if pd.isna(precip_val):
                precip_val = 0.0  # Replace NaN with 0
            precip_values.append(f"{precip_val:.2f}")
        
        # Create GAGES line
        precip_str = "  ".join(precip_values)
        gssha_content.append(f"GAGES {date_str}  {precip_str}")
    
    # Write to file
    start_year_month = pd.to_datetime(flood_start_date).strftime('%Y%m')
    end_year_month = pd.to_datetime(flood_end_date).strftime('%Y%m')
    gssha_filename = f"rainfall_input_{start_year_month}_{end_year_month}.gag"
    gssha_filepath = os.path.join(output_location, gssha_filename)
    
    with open(gssha_filepath, 'w') as f:
        for line in gssha_content:
            f.write(line + '\n')
    
    print(f"\n💾 GSSHA RAINFALL INPUT FILE CREATED:")
    print(f"  File: {gssha_filepath}")
    print(f"  Size: {len(gssha_content)} lines")
    
    # Display first few lines as preview
    print(f"\n📄 PREVIEW (first 15 lines):")
    print("-" * 70)
    for i, line in enumerate(gssha_content[:15], 1):
        print(f"{i:3d}: {line}")
    
    if len(gssha_content) > 15:
        print(f"     ... ({len(gssha_content) - 15} more lines)")
    
    # Summary statistics
    print(f"\n📈 DATA SUMMARY:")
    station_stats = []
    for station_col in station_coords.keys():
        station_id = station_col.split('_')[0]
        total_precip = flood_data[station_col].sum()
        max_precip = flood_data[station_col].max()
        avg_precip = flood_data[station_col].mean()
        
        station_stats.append({
            'Station': station_id,
            'Total (mm)': f"{total_precip:.2f}",
            'Max (mm)': f"{max_precip:.2f}",
            'Average (mm)': f"{avg_precip:.3f}"
        })
    
    stats_df = pd.DataFrame(station_stats)
    print(stats_df.to_string(index=False))
    
    return gssha_filepath

# Execute the GSSHA conversion
if not flood_data_complete.empty:
    print("Starting GSSHA rainfall input conversion...")
    
    # Use the target projection from the variables (if available, otherwise use default)
    target_proj = 'EPSG:32645' if 'Target_projection' not in globals() else Target_projection
    
    gssha_file = create_gssha_rainfall_input(
        flood_data_complete,
        metadata_within_bounds, 
        target_proj,
        output_location,
        flood_start_date,
        flood_end_date
    )
    
    if gssha_file:
        print(f"\n🎉 SUCCESS: GSSHA rainfall input file created!")
        print(f"📁 File ready for GSSHA modeling: {gssha_file}")
    else:
        print(f"\n❌ FAILED: Could not create GSSHA input file")
else:
    print("❌ No flood data available. Please run the previous cell to create flood period data first.")

Starting GSSHA rainfall input conversion...
🔧 CONVERTING TO GSSHA RAINFALL INPUT FORMAT
📍 Station Coordinates (converted to EPSG:32645):
------------------------------------------------------------
  TP001    |    331346.54   3069353.42 | Nagarjun (Prativa's house)
  TP002    |    335315.79   3072831.61 | Tokha (Ashish Dangol's house
  TP003    |    350294.07   3070053.60 | Lapsephedi
  TP005    |    333728.04   3061377.09 | Kusunti Office
  TP006    |    346054.81   3061759.11 | Bhaktapur (KEC)
  TP007    |    340124.57   3048330.45 | Bhardev
  TP008    |    344556.68   3075663.76 | Okhreni Tipping Bucket

📊 GSSHA Parameters:
  Number of rain gages (NRGAG): 7
  Number of time periods (NRPDS): 385
  Time range: 2023-08-06 00:00:00 to 2023-08-10 00:00:00

📝 Creating GAGES cards for 385 time periods...

💾 GSSHA RAINFALL INPUT FILE CREATED:
  File: ../../Results/Floods/Flood_Event\rainfall_input_202308_202308.gag
  Size: 395 lines

📄 PREVIEW (first 15 lines):
-----------------------------

In [70]:
# Check for periods with actual precipitation in the dataset
print("🔍 FINDING PERIODS WITH ACTUAL PRECIPITATION")
print("=" * 60)

# Remove comma from date strings if present
clean_start_date =flood_start_date.lstrip(',')
clean_end_date = flood_end_date.lstrip(',')

print(f"Cleaned dates:")
print(f"  Start: {clean_start_date}")
print(f"  End: {clean_end_date}")

# Check precipitation data for periods with rain
station_cols = [col for col in precipitation_data.columns if col != 'Date and Time']

print(f"\n📊 CHECKING OVERALL PRECIPITATION DATA:")
print(f"Total precipitation records: {len(precipitation_data):,}")
print(f"Date range: {precipitation_data['Date and Time'].min()} to {precipitation_data['Date and Time'].max()}")

# Find periods with significant precipitation
print(f"\n🌧️ PERIODS WITH PRECIPITATION > 0:")
print("-" * 50)

for station in station_cols[:3]:  # Check first 3 stations
    station_id = station.split('_')[0]
    rain_data = precipitation_data[precipitation_data[station] > 0]
    
    if not rain_data.empty:
        print(f"\n{station_id}:")
        print(f"  Total rainy records: {len(rain_data)}")
        print(f"  Date range with rain: {rain_data['Date and Time'].min()} to {rain_data['Date and Time'].max()}")
        print(f"  Max precipitation: {rain_data[station].max():.2f} mm")
        
        # Show a few examples of rainy periods
        print(f"  Sample rainy periods:")
        sample_rain = rain_data.head(5)
        for _, row in sample_rain.iterrows():
            print(f"    {row['Date and Time']}: {row[station]:.2f} mm")
    else:
        print(f"{station_id}: No precipitation > 0 found")

# Suggest using a period with actual rain
print(f"\n💡 SUGGESTION:")
print("The current period (April 19-23, 2024) appears to be a dry period.")
print("Consider using a period with actual precipitation for meaningful analysis.")

🔍 FINDING PERIODS WITH ACTUAL PRECIPITATION
Cleaned dates:
  Start: 2023-08-06 00:00:00
  End: 2023-08-10 00:00:00

📊 CHECKING OVERALL PRECIPITATION DATA:
Total precipitation records: 247,626
Date range: 2018-01-01 00:00:00 to 2025-08-29 11:00:00

🌧️ PERIODS WITH PRECIPITATION > 0:
--------------------------------------------------

TP001:
  Total rainy records: 8768
  Date range with rain: 2018-04-16 00:00:00 to 2025-03-03 13:30:00
  Max precipitation: 23.60 mm
  Sample rainy periods:
    2018-04-16 00:00:00: 0.20 mm
    2018-04-16 18:00:00: 0.20 mm
    2018-04-17 00:00:00: 0.20 mm
    2018-04-17 00:15:00: 0.20 mm
    2018-04-17 00:30:00: 0.20 mm

TP002:
  Total rainy records: 11017
  Date range with rain: 2018-04-16 00:00:00 to 2025-08-29 09:15:00
  Max precipitation: 26.80 mm
  Sample rainy periods:
    2018-04-16 00:00:00: 0.20 mm
    2018-04-16 16:15:00: 0.20 mm
    2018-04-17 00:30:00: 0.20 mm
    2018-04-17 00:45:00: 0.20 mm
    2018-04-17 01:15:00: 0.20 mm

TP003:
  Total rainy